In [ ]:
import os
import numpy as np
import pandas as pd


def compute_table2_metrics(csv_path, model_name=None):
  """Reads a single K-Fold CSV file and computes Mean ± Std for Table 2.

  Args:
      csv_path (str): Path to the CSV file (e.g., 'unet_1000.csv',
        'resnet_1000.csv').
      model_name (str, optional): Name of the model to display. If None, it is
        inferred from the filename.

  Returns:
      pd.DataFrame: A single-row DataFrame formatted as Mean ± Std.
      str: Formatted LaTeX table row.
  """
  if not os.path.exists(csv_path):
    raise FileNotFoundError(f"File not found: {csv_path}")

  df = pd.read_csv(csv_path)

  # Infer model name if not provided
  if model_name is None:
    filename = os.path.splitext(os.path.basename(csv_path))[0]
    model_name = (
        filename.replace('_1000', '').replace('_1800', '').replace('_', '-')
    )

  # 1. Filter out non-numeric summary rows (e.g., 'AVERAGE' or 'mean')
  fold_cols = [
      c for c in df.columns if c.lower() in ['fold', 'kfold', 'folds', 'split']
  ]
  if fold_cols:
    fold_col = fold_cols[0]
    df = df[pd.to_numeric(df[fold_col], errors='coerce').notnull()].copy()

  # 2. Case-insensitive mapping for validation columns
  metric_patterns = {
      'Accuracy': ['val_accuracy', 'val_accu', 'val_acc'],
      'Dice': ['val_dice'],
      'IoU': ['val_iou', 'val_jaccard'],
      'Precision': ['val_precision', 'val_prec'],
      'Recall': ['val_recall', 'val_reca', 'val_rec', 'val_sensitivity'],
      'F1-Score': ['val_f1', 'val_f1_score', 'val_f1-score'],
  }

  lower_to_orig = {col.lower(): col for col in df.columns}
  metrics_summary = {}

  for metric, candidates in metric_patterns.items():
    matched_col = None
    for cand in candidates:
      if cand in lower_to_orig:
        matched_col = lower_to_orig[cand]
        break

    # In binary segmentation, F1-Score is mathematically identical to Dice
    if matched_col is None and metric == 'F1-Score':
      for cand in metric_patterns['Dice']:
        if cand in lower_to_orig:
          matched_col = lower_to_orig[cand]
          break

    if matched_col is not None:
      series = pd.to_numeric(df[matched_col], errors='coerce').dropna()
      mean = series.mean()
      std = series.std(ddof=1)  # Sample standard deviation across folds
      metrics_summary[metric] = {
          'mean': mean,
          'std': std,
          'text': f'{mean:.4f} ± {std:.4f}',
          'latex': f'{mean:.4f} \\pm {std:.4f}',
      }
    else:
      metrics_summary[metric] = {
          'mean': np.nan,
          'std': np.nan,
          'text': 'N/A',
          'latex': 'N/A',
      }

  # 3. Create the formatted single-row DataFrame
  table_row = {
      'Model': model_name,
      'Accuracy': metrics_summary['Accuracy']['text'],
      'Dice': metrics_summary['Dice']['text'],
      'IoU': metrics_summary['IoU']['text'],
      'Precision': metrics_summary['Precision']['text'],
      'Recall': metrics_summary['Recall']['text'],
      'F1-Score': metrics_summary['F1-Score']['text'],
  }
  result_df = pd.DataFrame([table_row])

  # 4. Generate LaTeX code
  latex_cells = [
      metrics_summary[m]['latex']
      for m in ['Accuracy', 'Dice', 'IoU', 'Precision', 'Recall', 'F1-Score']
  ]
  latex_row = f"{model_name} & {' & '.join(latex_cells)} \\\\"

  return result_df, latex_row


# ========================================================
# Usage Example: Run on a single CSV file
# ========================================================
if __name__ == '__main__':
  # Specify your CSV file path here:
  csv_file = 'unet_1000.csv'  # Can also be 'resnet_1000.csv' or 'segnet_1000.csv'

  # Compute metrics
  table_row_df, latex_code = compute_table2_metrics(
      csv_file, model_name='U-Net'
  )

  print('=' * 85)
  print('Table 2 Row Output:')
  print(table_row_df.to_string(index=False))
  print('=' * 85)
  print('\nLaTeX code:')
  print(latex_code)